In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

DATA_DIR = CURRENT_DIR.parent / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'

RAW_DATA_FILE = RAW_DATA_DIR / f'complete_pitch_log.csv'
PA_RESULT_FILE = PROCESSED_DATA_DIR / f'pa_results.csv'
LINE_SCORE_FILE = PROCESSED_DATA_DIR / f'line_scores.csv'
GAME_SCORES_FILE = PROCESSED_DATA_DIR / f'player_game_scores.csv'

In [ ]:
statcast_df = pd.read_csv(RAW_DATA_FILE, low_memory=False)
pa_df = pd.read_csv(PA_RESULT_FILE, low_memory=False)
ls_df= pd.read_csv(LINE_SCORE_FILE, low_memory=False)

In [8]:
"""
- Per-player, per-game game score from the full Statcast pitch log.
- Accounts every pitch seen by the batter in a game, accounting for results, quality of contact, swing/miss metrics, pitch metrics
- NEXT STEPS:
    - Add more granular pitch metrics to the scoring algorithm, including pitch location, movement, and spin rate.
    - Consider adding weights for different pitch types and their effectiveness against specific batters.
    - Develop a rolling scoring system that accounts for a player's performance over a given span of games
        - Will be useful for evaluating performance trends and using them in predictive modeling
"""
def calculate_game_score(statcast_df, scale=100.0):
    if statcast_df is None or statcast_df.empty:
        return pd.DataFrame(
            columns=[
                "game_pk",
                "game_date",
                "player_id",
                "player_name",
                "team",
                "opposing_team",
                "game_score_raw",
                "game_score",
            ]
        )

    pitches = statcast_df.copy()

    # Basic game / player identifiers
    pitches["player_id"] = pd.to_numeric(pitches["batter"], errors="coerce").astype("Int64")
    pitches["player_name"] = pitches["player_name"].fillna("")
    pitches["game_date"] = pd.to_datetime(pitches["game_date"], errors="coerce").dt.strftime("%Y-%m-%d")

    # Relevant contextual pitch/situation metrics
    for col in [
        "release_speed",
        "release_spin_rate",
        "launch_speed",
        "hit_distance_sc",
        "launch_angle",
        "bat_speed",
        "swing_length",
        "miss_distance",
        "woba_value",
        "delta_run_exp",
        "on_1b",
        "on_2b",
        "on_3b",
        "outs_when_up",
        "inning",
        "arm_angle",
        "pfx_x",
        "pfx_z",
        "api_break_x_arm",
        "api_break_z_with_gravity",
    ]:
        if col in pitches.columns:
            pitches[col] = pd.to_numeric(pitches[col], errors="coerce")

    pitches["pitch_type_id"] = pd.to_numeric(pitches["pitch_type_id"], errors="coerce").fillna(0)

    pitches["runners_on"] = 0.0
    for base_col in ["on_1b", "on_2b", "on_3b"]:
        if base_col in pitches.columns:
            pitches["runners_on"] += pd.to_numeric(pitches[base_col], errors="coerce").fillna(0)

    if "events" in pitches.columns:
        pitches["events_clean"] = pitches["events"].fillna("").str.lower().str.replace(" ", "_", regex=False)
    else:
        pitches["events_clean"] = ""

    if "description" in pitches.columns:
        pitches["description_clean"] = pitches["description"].fillna("").str.lower()
    else:
        pitches["description_clean"] = ""

    # Outcome weights based on pitch result and quality of contact
    event_map = {
        "single": 15,
        "double": 30,
        "triple": 45,
        "home_run": 60,
        "walk": 15,
        "hit_by_pitch": 10,
        "strikeout": -20,
        "field_out": -10,
        "grounded_into_double_play": -30,
        "double_play": -30,
        "sac_fly": 8,
        "sac_bunt": 6,
        "force_out": -6,
        "lineout": -5,
        "flyout": -4,
        "error": 5,
        "fielders_choice": 5,
        "batter_interference": 5,
        "intent_walk": 15,
    }

    def pitch_score(row):
        score = 0.0

        event = row["events_clean"]
        desc = row["description_clean"]
        score += event_map.get(event, 0)

        # Pitch-specific stuff / shape adjustments
        #score += float(row.get("pitch_type_id", 0)) * 0.8

        if pd.notna(row.get("pfx_z")):
            score += max(0.0, abs(float(row["pfx_z"])) - 5) * 0.2
        if pd.notna(row.get("pfx_x")):
            score += max(0.0, abs(float(row["pfx_x"])) - 5) * 0.2

        #if pd.notna(row.get("woba_value")):
        #    score += float(row["woba_value"]) * 22

        if pd.notna(row.get("launch_speed")):
            score += max(0.0, (float(row["launch_speed"]) - 88) * 0.5) # League average approx 88 mph

        # TO DO: TUNE WEIGHTS FOR CONTACT TYPES
        if pd.notna(row.get("bb_type")) and row["bb_type"] == "ground_ball":
            if pd.notna(row.get("hit_distance_sc")):
                score += max(0.0, (float(row["hit_distance_sc"]) - 30) * 0.05) # Calculated average approx 30 ft for ground balls

        if pd.notna(row.get("bb_type")) and row["bb_type"] == "popup":
            if pd.notna(row.get("hit_distance_sc")):
                score += max(0.0, (float(row["hit_distance_sc"]) - 125) * 0.05) # Calculated average approx 125 ft for popups

        if pd.notna(row.get("bb_type")) and row["bb_type"] == "line_drive":
            if pd.notna(row.get("hit_distance_sc")):
                score += max(0.0, (float(row["hit_distance_sc"]) - 250) * 0.05) # Calculated average approx 250 ft for line drives

        if pd.notna(row.get("bb_type")) and row["bb_type"] == "fly_ball":
            if pd.notna(row.get("hit_distance_sc")):
                score += max(0.0, (float(row["hit_distance_sc"]) - 315) * 0.05) # Calculated average approx 315 ft for fly balls

        if pd.notna(row.get("delta_run_exp")):
            score += float(row["delta_run_exp"]) * 15

        # Situational value
        runners_on = row.get("runners_on", 0)
        outs = row.get("outs_when_up", 0)
        inning = row.get("inning", 0)

        score += float(runners_on) * 2.5
        if outs >= 2:
            score += 4
        if inning >= 7:
            score += 3

        # Chase / poor swing penalties
        # DETERMINE AVERAGES / WEIGHTS
        if any(token in desc for token in ["swinging_strike", "called_strike", "missed_bunt"]):
            score -= 6

        if pd.notna(row.get("miss_distance")):
            miss_distance = float(row["miss_distance"])
            score -= max(0.0, (miss_distance - 8) * 0.8)

        if pd.notna(row.get("bat_speed")):
            bat_speed = float(row["bat_speed"])
            if bat_speed < 60:
                score -= 2

        # Pitch quality / velocity and spin considerations
        # DETERMINE AVERAGES / WEIGHTS
        if pd.notna(row.get("release_speed")):
            score += max(0.0, (float(row["release_speed"]) - 87) * 0.2)

        if pd.notna(row.get("release_spin_rate")):
            score += max(0.0, (float(row["release_spin_rate"]) - 1800) / 200)

        return score

    pitches["pitch_score"] = pitches.apply(pitch_score, axis=1)

    game_score_df = (
        pitches.groupby(["game_pk", "game_date", "player_id", "player_name", "team"], dropna=False, as_index=False)["pitch_score"]
        .sum()
        .rename(columns={"pitch_score": "game_score_raw"})
    )

    if "home_team" in pitches.columns and "away_team" in pitches.columns:
        lookup = pitches[["game_pk", "team", "home_team", "away_team"]].drop_duplicates()
        game_score_df = game_score_df.merge(lookup, on=["game_pk", "team"], how="left")
        game_score_df["opposing_team"] = np.where(
            game_score_df["team"] == game_score_df["home_team"],
            game_score_df["away_team"],
            game_score_df["home_team"],
        )
    else:
        game_score_df["opposing_team"] = np.nan

    # Normalize raw total scores
    game_score_df["game_score"] = 0.0
    if not game_score_df.empty:
        raw_min = game_score_df["game_score_raw"].min()
        raw_max = game_score_df["game_score_raw"].max()
        if raw_max > raw_min:
            game_score_df["game_score"] = (game_score_df["game_score_raw"] - raw_min) / (raw_max - raw_min) * scale
        game_score_df["game_score"] = game_score_df["game_score"].clip(lower=0, upper=scale)

    return game_score_df[["game_pk", "game_date", "player_id", "player_name", "team", "opposing_team", "game_score_raw", "game_score"]]

# Use the full pitch-level Statcast dataframe so every pitch seen in a game contributes
player_game_scores = calculate_game_score(statcast_df)

if not player_game_scores.empty:
    player_game_scores = (
        ls_df[["game_pk", "game_date", "player_id", "player_name", "team", "opposing_team", "season"]]
        .merge(player_game_scores, on=["game_pk", "game_date", "player_id", "player_name", "team"], how="left")
    )
    player_game_scores["game_score"] = player_game_scores["game_score"].fillna(0).clip(lower=0, upper=100)
    player_game_scores = player_game_scores.sort_values(["game_date", "player_name", "game_pk"]).reset_index(drop=True)
    display(player_game_scores.head(10))
    player_game_scores.to_csv(GAME_SCORES_FILE, index=False, encoding='utf-8')
else:
    print("INFO: No game scores were generated from the pitch log.")


,game_pk,game_date,player_id,player_name,team,opposing_team_x,season,opposing_team_y,game_score_raw,game_score
0,778555,2025-03-27,682928,"Abrams, CJ",WSH,PHI,2025,PHI,70.660000,28.416660
1,778555,2025-03-27,605137,"Bell, Josh",WSH,PHI,2025,PHI,66.630000,27.692007
2,778557,2025-03-27,641355,"Bellinger, Cody",NYY,MIL,2025,MIL,139.925000,40.871527
3,778557,2025-03-27,665828,"Cabrera, Oswaldo",NYY,MIL,2025,MIL,74.780000,29.157497
4,778557,2025-03-27,665862,"Chisholm Jr., Jazz",NYY,MIL,2025,MIL,27.864913,20.721474
5,778555,2025-03-27,686611,"Crews, Dylan",WSH,PHI,2025,PHI,102.535000,34.148255
6,778555,2025-03-27,657557,"DeJong, Paul",WSH,PHI,2025,PHI,-9.010000,14.090822
7,778557,2025-03-27,691176,"Domínguez, Jasson",NYY,MIL,2025,MIL,3.140000,16.275571
8,778555,2025-03-27,671277,"García Jr., Luis",WSH,PHI,2025,PHI,117.896381,36.910457
9,778557,2025-03-27,502671,"Goldschmidt, Paul",NYY,MIL,2025,MIL,-3.460000,15.088794
